# Lab 1 — MLflow Experiment Tracking
Run only this notebook for Lab 1. Its prerequisite model preparation is included. Read START_HERE.md for Docker/UI tasks and submission instructions.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, json, time, shutil
uploaded = files.upload()  # Choose Lab1_MLflow_Experiment_Tracking.zip
zip_name = next(name for name in uploaded if name.endswith('.zip'))
base = Path('/content/cartvista-lab1')
base.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    # Reject path traversal; only extract the supplied source package.
    for name in z.namelist():
        if not (base / name).resolve().is_relative_to(base.resolve()):
            raise ValueError('Unsafe ZIP entry')
    z.extractall(base)
PROJECT = base / 'Lab1_MLflow_Experiment_Tracking'
assert (PROJECT / 'requirements.txt').exists()
os.chdir(PROJECT)
print('Project:', PROJECT)


## Install isolated lab environment
The older MLflow/sklearn lab versions need Python 3.11/3.12. The separate environment avoids downgrading Colab's notebook packages. On a future newer hosted Python, this cell obtains a separate Python 3.11 using uv. Setup can take several minutes.


In [ ]:
AUTHOR = 'Sai YourFullName'  # CHANGE THIS BEFORE TRAINING
os.environ['LAB_AUTHOR'] = AUTHOR
VENV = Path('/content/cartvista-lab1-venv')
if sys.version_info[:2] in [(3,11),(3,12)]:
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(VENV)], check=True)
PY = str(VENV / 'bin/python')
subprocess.run([PY, '-m', 'pip', 'install', '-r', str(PROJECT/'requirements.txt')], check=True)
def run(*args):
    p = subprocess.Popen([PY, '-u', *args], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines=[]
    for line in p.stdout:
        print(line, end=''); lines.append(line)
    status=p.wait()
    if status: raise subprocess.CalledProcessError(status, args, output=''.join(lines))
    return ''.join(lines)
run('-c', 'import mlflow,sklearn; print(mlflow.__version__,sklearn.__version__)')


## Lab 1 All experiments and exercises
This runs the six-candidate comparison, precision/recall logging, author tags, two gradient boosting candidates, registry versions, consumer, safe one-row fingerprint experiment and autologging. A repeated run adds new MLflow runs. Inspect your actual winner; sample outcomes in the brief are not guaranteed.


In [ ]:
run('run_lab1.py')


### MLflow interface
The browser viewer may depend on Colab's current proxy behavior. If the UI does not open, use the printed run table below and collect the UI screenshots on the Docker route. Do not interrupt training while downloading the SQLite store.


In [ ]:
UI = subprocess.Popen([str(VENV/'bin/mlflow'), 'ui', '--backend-store-uri', 'sqlite:///'+str(PROJECT/'mlflow.db'), '--host', '0.0.0.0', '--port', '5000'], cwd=PROJECT, stdout=open('/content/cartvista-mlflow.log','w'), stderr=subprocess.STDOUT)
time.sleep(5)
from google.colab import output
output.serve_kernel_port_as_window(5000)


In [ ]:
run('-c', "from common import setup,EXPERIMENT; c=setup(); e=c.get_experiment_by_name(EXPERIMENT); rows=c.search_runs([e.experiment_id],order_by=['metrics.test_auc DESC']); print('RUN ID | MODEL | AUC | AUTHOR'); [print(r.info.run_id,r.data.params.get('model_kind'),r.data.metrics.get('test_auc'),r.data.tags.get('author')) for r in rows]")


## Download Lab 1 evidence
Save this executed notebook separately. Complete this lab’s report and any Docker/UI evidence described in START_HERE.md.

In [ ]:
if UI.poll() is None: UI.terminate(); UI.wait(timeout=15)
stage=Path('/content/lab1-submission-evidence')
if stage.exists(): shutil.rmtree(stage)
stage.mkdir()
source=PROJECT/'evidence'
if source.exists():
    for f in source.rglob('*'):
        if not f.is_file(): continue
        # Include only this lab's results, plus relevant Colab-specific metrics.
        if f.name.startswith('lab1_') or False and (f.name.startswith('colab_') or 'pipeline' in f.parts or f.name=='gate_failures.log') or False and f.name.startswith('metrics_'):
            d=stage/'evidence'/f.relative_to(source);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(f,d)
for name in ['churn_data.csv']:
    if (PROJECT/name).exists(): shutil.copy2(PROJECT/name,stage/name)
if (PROJECT/'model_export').exists(): shutil.copytree(PROJECT/'model_export',stage/'model_export')
archive=shutil.make_archive('/content/Lab1_My_Run_Evidence','zip',stage)
files.download(archive)
print('Finish the separate Lab 1 report and save this executed notebook.')
